# RAG Latency Benchmark

In this notebook, we'll build a custom benchmark to see exactly where our RAG pipeline spends its time. We'll do it step-by-step to gain deep knowledge of the retrieval mechanics.

In [1]:
import os
import time 
import nest_asyncio
import numpy as np
from dotenv import load_dotenv

In [2]:
load_dotenv()
nest_asyncio.apply()

In [3]:
import sys
sys.path.append('..')
from pure_rag.chunking_utils import sliding_window_chunker
from pure_rag.retrieval_utils import SimpleRetriever

In [4]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings, VectorStoreIndex, Document

print("Setup Ready!")

Setup Ready!


In [5]:
test_text= "LlamaIndex is a data framework for LLM applications. It helps bridge private data with public models. " * 30

query_text= "What is LlamaIndex?"

print(f"Test text length: {len(test_text)} characters")

Test text length: 3060 characters


## Pure RAG (Manual) Timing

In [6]:
import google.generativeai as genai

print("--- Starting Pure RAG Benchmark ---")

genai.configure(api_key= os.getenv("GEMINI_API_KEY"))

--- Starting Pure RAG Benchmark ---


C:\Users\ASUS\AppData\Local\Temp\ipykernel_6412\2251631649.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [7]:
# 1.TIMING: CHUNKING
t0= time.time()
chunks= sliding_window_chunker(test_text, size= 200, overlap= 50)
t_chunk= time.time() - t0
print(f"1. Chunking Time: {t_chunk: .4f}s") 

1. Chunking Time:  0.0000s


In [8]:
# 2. TIMING: EMBEDDING (THE API CALLS)
t0= time.time()
embeddings= []
for chunk in chunks:
    res= genai.embed_content(model="models/gemini-embedding-001", content=chunk)
    embeddings.append(res['embedding'])
t_embed= time.time() - t0
print(f"2. Embedding Time: {t_embed: .4f}s (Serial loop)")

2. Embedding Time:  9.6219s (Serial loop)


In [9]:
# 3. TIMING: RETRIEVAL (Search math)
t0 = time.time()

# 1. Initialize with our objects
retriever = SimpleRetriever(embeddings, chunks) 

# 2. Get the query embedding (API call)
q_res = genai.embed_content(model="models/gemini-embedding-001", content=query_text)
q_embed = q_res['embedding']

# 3. Search for the best match (This creates best_chunk!)
best_chunk = retriever.search(q_embed, k=1)

t_search = time.time() - t0
print(f"3. Retrieval Time: {t_search:.4f}s")

3. Retrieval Time: 0.4761s


In [10]:
t0= time.time()

prompt= f"Use time context to answer: {best_chunk}\nQuestion: {query_text}"

model= genai.GenerativeModel("models/gemini-2.0-flash")
response= model.generate_content(prompt)

t_gen= time.time() - t0
print(f"4. Generation Time: {t_gen: .4f}s")

pure_total= t_chunk + t_embed + t_search + t_gen
print(f"\n--- TOTAL PURE RAG: {pure_total: .4f}s ---")

4. Generation Time:  1.0453s

--- TOTAL PURE RAG:  11.1433s ---


##  LlamaIndex Timing

In [11]:
Settings.llm = GoogleGenAI(model="models/gemini-2.0-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="models/gemini-embedding-001")

print("LlamaIndex Configured for Gemini! ")

LlamaIndex Configured for Gemini! 


In [12]:
print("--- Starting LlamaIndex Benchmark ---")

# Step 0: Wrap our test text in a Document object
doc= Document(text= test_text)

--- Starting LlamaIndex Benchmark ---


In [13]:

# 1. TIMING: INDEXING (Chunking + Embedding combined)
# This is usually where LlamaIndex shines because it batches the chunks!
t0= time.time()
index= VectorStoreIndex.from_documents([doc])
t_index= time.time() - t0
print(f"1. Indexing Time: {t_index: .4f}s (Chunk + Embed)")

1. Indexing Time:  0.7142s (Chunk + Embed)


In [14]:
# 2. TIMING: RETRIEVAL (Search)
t0 = time.time()
retriever = index.as_retriever(similarity_top_k=1)
results = retriever.retrieve(query_text)
t_retrieve_li = time.time() - t0
print(f"2. Retrieval Time:   {t_retrieve_li:.4f}s")

2. Retrieval Time:   0.4777s


In [15]:
# 3. TIMING: QUERYING (Synthesis)
t0 = time.time()
query_engine= index.as_query_engine()
response= query_engine.query(query_text)
t_gen_li = time.time() - t0
print(f"3. Generation Time:  {t_gen_li:.4f}s")

3. Generation Time:  1.5026s


In [16]:
li_total = t_index + t_retrieve_li + t_gen_li
print(f"\nTOTAL LLAMAINDEX:    {li_total:.4f}s")


TOTAL LLAMAINDEX:    2.6945s
